# 04. Auditoría LLM y Match Final

## Goal
Construir el mapeo final de empresas priorizando evidencia determinística: razón social Ecuador validada manualmente, matches exactos contra catálogos, y auditoría LLM solo como apoyo para candidatos fuzzy no exactos.


## Inputs
- `outputs/leads_ruc_exact.csv`
- `outputs/horas_ruc_exact.csv`
- `outputs/leads_torneo_ganadores.csv`
- `outputs/horas_torneo_ganadores.csv`
- `outputs/empresas_universe_compilado.csv`

## Outputs
- `outputs/audit_df.csv`: candidatos fuzzy con evaluación determinística y, si aplica, LLM.
- `outputs/candidatos_revision_empresas.csv`: todos los candidatos no determinísticos para revisar.
- `outputs/empresas_sin_match_final.csv`: entidades no aceptadas con motivo.
- `outputs/match_final_empresas.csv`: mapeo final aceptado con RUC como texto.


In [1]:
# ── Helpers y rutas ──────────────────────────────────────────────────────────
from pathlib import Path
import os, json, re
import pandas as pd
import urllib.request, urllib.error

try:
    from IPython.display import display
except Exception:
    def display(x): print(x)


def find_project_root(start=None):
    start = (start or Path.cwd()).resolve()
    for p in [start, *start.parents]:
        if (p / "01_data_ingestion_enrichment").is_dir() and (p / "02_data_cleaning").is_dir():
            return p
    raise FileNotFoundError(f"No se pudo localizar la raíz. cwd: {start}")


def ensure_dir(d):
    Path(d).mkdir(parents=True, exist_ok=True)
    return Path(d)


def save_df_csv(df, path, *, index=False, encoding="utf-8-sig"):
    path = Path(path)
    ensure_dir(path.parent)
    df.to_csv(path, index=index, encoding=encoding)
    print(f"[OK] Guardado: {path.resolve()}  shape: {df.shape}")
    return path


def read_csv_checked(path, **kwargs):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"No existe: {path.resolve()}")
    return pd.read_csv(path, **kwargs)


def normalize_ruc(value) -> str:
    if value is None or pd.isna(value):
        return ""
    s = str(value).strip()
    if not s:
        return ""
    if re.fullmatch(r"\d+\.0", s):
        s = s[:-2]
    digits = re.sub(r"\D+", "", s)
    # Solo se rellena el caso común de un RUC de 13 dígitos que perdió el cero inicial.
    return digits.zfill(13) if len(digits) == 12 else digits


def is_valid_ruc(value) -> bool:
    ruc = normalize_ruc(value)
    return len(ruc) == 13 and ruc.isdigit()


def to_bool(value) -> bool:
    if value is None or pd.isna(value):
        return False
    if isinstance(value, bool):
        return value
    s = str(value).strip().upper()
    return s in {"TRUE", "1", "1.0", "SI", "S", "YES", "Y"}


def clean_text(value) -> str:
    if value is None or pd.isna(value):
        return ""
    return str(value).strip()


ROOT = find_project_root()
CLEAN_OUT = ROOT / "02_data_cleaning" / "outputs"
print(f"[CONFIG] ROOT      : {ROOT}")
print(f"[CONFIG] CLEAN_OUT : {CLEAN_OUT}")


[CONFIG] ROOT      : E:\TESIS MAESTRIA\Desarrollo_clustering_maestria
[CONFIG] CLEAN_OUT : E:\TESIS MAESTRIA\Desarrollo_clustering_maestria\02_data_cleaning\outputs


In [2]:
# ── Cargar entradas ──────────────────────────────────────────────────────────
# Importante: RUC se lee siempre como texto para no perder ceros iniciales.
leads_torneo_path = CLEAN_OUT / "leads_torneo_ganadores.csv"
horas_torneo_path = CLEAN_OUT / "horas_torneo_ganadores.csv"
leads_manual_exact_path = CLEAN_OUT / "leads_manual_razon_ruc_exact.csv"
horas_manual_exact_path = CLEAN_OUT / "horas_manual_razon_ruc_exact.csv"
leads_exact_path = CLEAN_OUT / "leads_ruc_exact.csv"
horas_exact_path = CLEAN_OUT / "horas_ruc_exact.csv"
universe_path = CLEAN_OUT / "empresas_universe_compilado.csv"

LEADS_TORNEO_COLS = ["Company_raw", "Company_norm", "source_winner", "winner_name_norm", "score", "RUC"]
HORAS_TORNEO_COLS = ["EMPRESA_raw", "EMPRESA_norm", "source_winner", "winner_name_norm", "score", "RUC"]
MANUAL_EXTRA_COLS = ["source_label", "source_winner", "winner_name_norm", "score", "manual_match_status", "manual_match_method", "manual_match_key", "manual_match_hits"]


def read_csv_optional(path, columns, *, reason="archivo opcional", **kwargs):
    path = Path(path)
    if not path.exists():
        print(f"[AVISO] No existe {path.name}; se continúa sin {reason}. Ejecuta 03 si quieres generarlo.")
        return pd.DataFrame(columns=columns)
    return pd.read_csv(path, **kwargs)


leads_exact = read_csv_checked(leads_exact_path, dtype={"RUC": "string"})
horas_exact = read_csv_checked(horas_exact_path, dtype={"RUC": "string"})
leads_manual_exact = read_csv_optional(leads_manual_exact_path, list(leads_exact.columns) + MANUAL_EXTRA_COLS, reason="match manual razón social", dtype={"RUC": "string"})
horas_manual_exact = read_csv_optional(horas_manual_exact_path, list(horas_exact.columns) + MANUAL_EXTRA_COLS, reason="match manual razón social", dtype={"RUC": "string"})
leads_torneo = read_csv_optional(leads_torneo_path, LEADS_TORNEO_COLS, reason="candidatos de torneo", dtype={"RUC": "string"})
horas_torneo = read_csv_optional(horas_torneo_path, HORAS_TORNEO_COLS, reason="candidatos de torneo", dtype={"RUC": "string"})
universe = read_csv_checked(universe_path)

for df in [leads_torneo, horas_torneo, leads_exact, horas_exact, leads_manual_exact, horas_manual_exact]:
    if "RUC" in df.columns:
        df["RUC"] = df["RUC"].map(normalize_ruc)


def warn_if_stale(output_path: Path, reference_paths: list[Path]) -> None:
    if not output_path.exists():
        return
    newer_refs = [rp.name for rp in reference_paths if rp.exists() and rp.stat().st_mtime > output_path.stat().st_mtime]
    if newer_refs:
        print(f"[AVISO] {output_path.name} es anterior a {', '.join(newer_refs)}; conviene regenerarlo ejecutando 03.")


for p in [leads_torneo_path, horas_torneo_path, leads_manual_exact_path, horas_manual_exact_path]:
    warn_if_stale(p, [leads_exact_path, horas_exact_path, universe_path])

print(f"[LEADS exact]         {leads_exact.shape}  RUC válidos: {int(leads_exact['RUC'].map(is_valid_ruc).sum())}")
print(f"[HORAS exact]         {horas_exact.shape}  RUC válidos: {int(horas_exact['RUC'].map(is_valid_ruc).sum())}")
print(f"[LEADS manual exact]  {leads_manual_exact.shape}  RUC válidos: {int(leads_manual_exact['RUC'].map(is_valid_ruc).sum()) if 'RUC' in leads_manual_exact.columns else 0}")
print(f"[HORAS manual exact]  {horas_manual_exact.shape}  RUC válidos: {int(horas_manual_exact['RUC'].map(is_valid_ruc).sum()) if 'RUC' in horas_manual_exact.columns else 0}")
print(f"[LEADS torneo]        {leads_torneo.shape}")
print(f"[HORAS torneo]        {horas_torneo.shape}")
print(f"[UNIVERSE]            {universe.shape}  cols: {universe.columns.tolist()}")


[LEADS exact]         (318, 12)  RUC válidos: 77
[HORAS exact]         (105, 12)  RUC válidos: 54
[LEADS manual exact]  (11, 20)  RUC válidos: 11
[HORAS manual exact]  (14, 20)  RUC válidos: 14
[LEADS torneo]        (158, 6)
[HORAS torneo]        (19, 6)
[UNIVERSE]            (420, 11)  cols: ['name_norm', 'name_raw', 'sources', 'n_rows', 'name_norm_antes_limpieza', 'name_raw_antes_limpieza', 'sources_antes_limpieza', 'n_rows_antes_limpieza', 'name_match_raw', 'razon_social_ecuador_raw', 'usa_razon_social_validada']


## Estrategia de decisión
1. La razón social Ecuador validada manualmente se conserva como nombre canónico de matching. La columna original (`Company_raw`/`EMPRESA_raw`) queda como referencia, no como sustituto de la razón social local.
2. `SCVS_EXACT` entra automáticamente porque proviene de cruce exacto por nombre normalizado y RUC válido.
3. Las razones sociales manuales sin RUC exacto no van al torneo: primero se cruzan determinísticamente contra SCVS/SRI por llaves legales casi crudas, compactas y normalizadas únicas.
4. Si la razón social manual apunta a un único RUC oficial, entra al final con evidencia `manual_razon_social_exact`.
5. Si no encuentra RUC o hay conflicto de RUC, queda en `*_manual_sin_ruc_revision.csv`.
6. El torneo fuzzy recibe solo huérfanos reales: registros sin RUC exacto y sin razón social Ecuador validada. Para ellos consulta con la primera columna normalizada.
7. `SRI_FANTASIA` queda desactivado por defecto en el torneo; si se necesita explorar nombres comerciales, se habilita explícitamente con `INCLUDE_SRI_FANTASIA=1`.
8. El LLM no genera RUC. Solo audita candidatos ya propuestos por catálogos oficiales; si no hay API key, el fuzzy no exacto queda para revisión.


In [3]:
# ── Construir candidatos de auditoría con reglas determinísticas ─────────────
AUDIT_THRESHOLD = float(os.getenv("UMBRAL_SRI", "80"))
AUDIT_MIN_CONFIDENCE = float(os.getenv("AUDIT_MIN_CONFIDENCE", "0.85"))
ALLOW_FUZZY_WITHOUT_LLM = os.getenv("ALLOW_FUZZY_WITHOUT_LLM", "0").strip() == "1"
AUTO_ACCEPT_EXACT_SOURCES = {"SCVS", "SRI_RAZON"}

COLS_FINAL_BASE = [
    "source_label", "name_raw", "name_norm", "RUC", "source_winner", "score", "verdict",
    "evidence_type", "razon_social_ecuador_raw", "usa_razon_social_validada",
    "sede_ecuador", "validacion_manual", "name_norm_original", "name_match_raw",
]


def meaningful_tokens(value: str) -> set:
    toks = re.findall(r"[A-Z0-9]+", str(value or "").upper())
    stop = {"EC", "ECUADOR", "CIA", "SA", "SAS", "LTDA", "COMPANIA", "COMPAÑIA"}
    return {t for t in toks if len(t) >= 2 and t not in stop}


def candidate_quality(row) -> tuple[str, str]:
    source_norm = clean_text(row.get("name_norm", ""))
    cand_norm = clean_text(row.get("winner_name_norm", ""))
    source_winner = clean_text(row.get("source_winner", "")).upper()
    score = float(row.get("score", 0) or 0)
    ruc = normalize_ruc(row.get("RUC", ""))
    manual_validated = to_bool(row.get("usa_razon_social_validada", False))
    razon_manual_llena = bool(clean_text(row.get("razon_social_ecuador_raw", "")))
    manual_name_present = manual_validated and razon_manual_llena
    sede_ecuador = to_bool(row.get("sede_ecuador", False))

    if not is_valid_ruc(ruc):
        return "reject", "RUC candidato no válido"
    if score < AUDIT_THRESHOLD:
        return "reject", f"Score menor al umbral {AUDIT_THRESHOLD:g}"
    if not source_norm or not cand_norm:
        return "reject", "Nombre fuente o candidato vacío"
    if "sede_ecuador" in row.index and not sede_ecuador and manual_name_present:
        return "reject", "Razón social manual indica que no aplica sede Ecuador"

    if source_norm == cand_norm:
        if manual_name_present and source_winner in AUTO_ACCEPT_EXACT_SOURCES:
            return "accept_exact_name", "Razón social manual validada coincide exactamente con catálogo confiable"
        return "needs_llm", "Coincidencia exacta de huérfano o fuente no prioritaria; requiere auditoría"

    src_tokens = meaningful_tokens(source_norm)
    cand_tokens = meaningful_tokens(cand_norm)
    overlap = src_tokens & cand_tokens

    if len(cand_tokens) <= 1 and len(src_tokens) > 1:
        return "reject", "Candidato demasiado genérico frente al nombre fuente"
    required_overlap = min(2, len(src_tokens), len(cand_tokens))
    if required_overlap and len(overlap) < required_overlap:
        return "reject", "Traslape insuficiente entre fuente y candidato"
    return "needs_llm", "Candidato plausible; requiere auditoría"


def enrich_torneo(torneo_df, exact_df, *, label, raw_col, norm_col):
    if torneo_df.empty:
        return pd.DataFrame()
    match_col = f"{raw_col.split('_')[0]}_match_raw" if "_" in raw_col else ""
    meta_cols = [
        raw_col, norm_col, f"{norm_col}_original", "razon_social_ecuador_raw",
        "usa_razon_social_validada", "sede_ecuador", "validacion_manual", match_col,
    ]
    meta_cols = [c for c in meta_cols if c and c in exact_df.columns]
    meta = exact_df[meta_cols].drop_duplicates([raw_col], keep="first")

    d = torneo_df.copy()
    d["RUC"] = d["RUC"].map(normalize_ruc)
    d[f"{norm_col}_torneo_original"] = d[norm_col] if norm_col in d.columns else ""
    d = d.merge(meta, on=raw_col, how="left", suffixes=("_torneo", ""))

    if norm_col not in d.columns:
        d[norm_col] = ""
    torneo_norm_col = f"{norm_col}_torneo"
    if torneo_norm_col in d.columns:
        d[norm_col] = d[norm_col].fillna(d[torneo_norm_col]).replace("", pd.NA).fillna(d[torneo_norm_col])

    d["source_label"] = label
    d["name_raw"] = d[raw_col]
    d["name_norm"] = d[norm_col]
    d["name_norm_torneo_original"] = d.get(f"{norm_col}_torneo_original", "")

    if "Company_norm_original" in d.columns:
        d["name_norm_original"] = d["Company_norm_original"]
    elif "EMPRESA_norm_original" in d.columns:
        d["name_norm_original"] = d["EMPRESA_norm_original"]
    else:
        d["name_norm_original"] = d["name_norm"]
    if "Company_match_raw" in d.columns:
        d["name_match_raw"] = d["Company_match_raw"]
    elif "EMPRESA_match_raw" in d.columns:
        d["name_match_raw"] = d["EMPRESA_match_raw"]
    else:
        d["name_match_raw"] = d["name_raw"]
    for c in ["razon_social_ecuador_raw"]:
        if c not in d.columns:
            d[c] = ""
    for c in ["usa_razon_social_validada", "sede_ecuador", "validacion_manual"]:
        if c not in d.columns:
            d[c] = False
        d[c] = d[c].map(to_bool)
    quality = d.apply(candidate_quality, axis=1, result_type="expand")
    d["deterministic_decision"] = quality[0]
    d["deterministic_reason"] = quality[1]
    d["audit_id"] = [f"{label}-{r.get('source_winner','')}-{i}" for i, r in d.iterrows()]
    return d


leads_candidates = enrich_torneo(leads_torneo, leads_exact, label="LEADS", raw_col="Company_raw", norm_col="Company_norm")
horas_candidates = enrich_torneo(horas_torneo, horas_exact, label="HORAS", raw_col="EMPRESA_raw", norm_col="EMPRESA_norm")
candidate_cols = [
    "audit_id", "source_label", "name_raw", "name_norm", "name_norm_torneo_original",
    "winner_name_norm", "RUC", "source_winner", "score", "deterministic_decision",
    "deterministic_reason", "razon_social_ecuador_raw", "usa_razon_social_validada",
    "sede_ecuador", "validacion_manual", "name_norm_original", "name_match_raw",
]
candidates_df = pd.concat([leads_candidates, horas_candidates], ignore_index=True)
for c in candidate_cols:
    if c not in candidates_df.columns:
        candidates_df[c] = pd.NA

auditable = candidates_df[candidates_df["deterministic_decision"].eq("needs_llm")].copy()
audit_cases = []
for _, r in auditable.iterrows():
    audit_cases.append({
        "id": r["audit_id"],
        "source_label": r["source_label"],
        "source_raw": clean_text(r.get("name_raw", "")),
        "source_norm": clean_text(r.get("name_norm", "")),
        "candidate_norm": clean_text(r.get("winner_name_norm", "")),
        "source_winner": clean_text(r.get("source_winner", "")),
        "score": float(r.get("score", 0) or 0),
        "ruc": normalize_ruc(r.get("RUC", "")),
        "razon_social_ecuador_raw": clean_text(r.get("razon_social_ecuador_raw", "")),
        "usa_razon_social_validada": bool(r.get("usa_razon_social_validada", False)),
    })

audit_df_preview = pd.DataFrame(audit_cases)
print(f"[CANDIDATOS] Total torneo: {len(candidates_df)}")
print(candidates_df["deterministic_decision"].value_counts(dropna=False).to_string())
print(f"[AUDITORIA] Casos enviados a LLM si hay API key: {len(audit_cases)}")
display(candidates_df.head(10))


[CANDIDATOS] Total torneo: 177
deterministic_decision
reject       140
needs_llm     37
[AUDITORIA] Casos enviados a LLM si hay API key: 37


,Company_raw,Company_norm_torneo,source_winner,winner_name_norm,score,RUC,Company_norm_torneo_original,Company_norm,Company_norm_original,razon_social_ecuador_raw,...,name_match_raw,deterministic_decision,deterministic_reason,audit_id,EMPRESA_raw,EMPRESA_norm_torneo,EMPRESA_norm_torneo_original,EMPRESA_norm,EMPRESA_norm_original,EMPRESA_match_raw
0,ACCO BRANDS,ACCO BRANDS,SRI_RAZON,BRANDS,100.0,0991336753001,ACCO BRANDS,ACCO BRANDS,ACCO BRANDS,NaN,...,ACCO BRANDS,reject,Candidato demasiado genérico frente al nombre ...,LEADS-SRI_RAZON-0,NaN,NaN,NaN,NaN,NaN,NaN
1,"ACH FOOD COMPANIES, INC",ACH FOOD COMPANIES,SRI_RAZON,FOOD,100.0,1792752310001,ACH FOOD COMPANIES,ACH FOOD COMPANIES,ACH FOOD COMPANIES,"ACH Food Companies, Inc.",...,"ACH FOOD COMPANIES, INC",reject,Candidato demasiado genérico frente al nombre ...,LEADS-SRI_RAZON-1,NaN,NaN,NaN,NaN,NaN,NaN
2,ALLIANZ MÉXICO,ALLIANZ MEXICO,SCVS,ALLIANZ,100.0,0195120420001,ALLIANZ MEXICO,ALLIANZ MEXICO,ALLIANZ MEXICO,NaN,...,ALLIANZ MÉXICO,reject,Candidato demasiado genérico frente al nombre ...,LEADS-SCVS-2,NaN,NaN,NaN,NaN,NaN,NaN
3,AFP Genesis,AFP GENESIS,SCVS,AFP GENESIS ADMINISTRADORA FONDOS FIDEICOMISOS,100.0,0991307605001,AFP GENESIS,AFP GENESIS,AFP GENESIS,NaN,...,AFP Genesis,needs_llm,Candidato plausible; requiere auditoría,LEADS-SCVS-3,NaN,NaN,NaN,NaN,NaN,NaN
4,AVERY DENNISON RBIS,AVERY DENNISON RBIS,SRI_RAZON,AVERY,100.0,0991367209001,AVERY DENNISON RBIS,AVERY DENNISON RBIS,AVERY DENNISON RBIS,NaN,...,AVERY DENNISON RBIS,reject,Candidato demasiado genérico frente al nombre ...,LEADS-SRI_RAZON-4,NaN,NaN,NaN,NaN,NaN,NaN
5,La Anita,ANITA,SCVS,COMERCIAL IMPORTADORA SANTA ANITA IMSANIT,100.0,0990085188001,ANITA,ANITA,ANITA,Productos Alimenticios La Anita S.A,...,La Anita,needs_llm,Candidato plausible; requiere auditoría,LEADS-SCVS-5,NaN,NaN,NaN,NaN,NaN,NaN
6,AXIONLOG,AXIONLOG,SCVS,AXIONLOG ECUADOR,100.0,0992991178001,AXIONLOG,AXIONLOG,AXIONLOG,AXIONLOG ECUADOR S.A.,...,AXIONLOG,needs_llm,Candidato plausible; requiere auditoría,LEADS-SCVS-6,NaN,NaN,NaN,NaN,NaN,NaN
7,BPL BIO PRODUCTS LABORATORY,BPL BIO PRODUCTS LABORATORY,SRI_RAZON,LABORATORY,100.0,1793196784001,BPL BIO PRODUCTS LABORATORY,BPL BIO PRODUCTS LABORATORY,BPL BIO PRODUCTS LABORATORY,NaN,...,BPL BIO PRODUCTS LABORATORY,reject,Candidato demasiado genérico frente al nombre ...,LEADS-SRI_RAZON-7,NaN,NaN,NaN,NaN,NaN,NaN
8,CFB,CFB,SCVS,CENTRO FERRETERO BAMBOO CFB,100.0,1793230370001,CFB,CFB,CFB,NaN,...,CFB,needs_llm,Candidato plausible; requiere auditoría,LEADS-SCVS-8,NaN,NaN,NaN,NaN,NaN,NaN
9,COMEX,COMEX,SCVS,ASOCIADOS EN SOLUCIONES EMPRESARIALES COMEX AS...,100.0,0993290599001,COMEX,COMEX,COMEX,NaN,...,COMEX,needs_llm,Candidato plausible; requiere auditoría,LEADS-SCVS-9,NaN,NaN,NaN,NaN,NaN,NaN


In [4]:
# ── Auditoría LLM opcional ──────────────────────────────────────────────────
# El LLM NO genera RUC. Solo audita si el candidato provisto es coherente.
# Modelo por defecto más fuerte que gpt-4o-mini; se puede cambiar con OPENAI_MODEL.

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY", "").strip()
OPENAI_MODEL = os.getenv("OPENAI_MODEL", "gpt-5.2").strip()
OPENAI_FALLBACK_MODELS = [m.strip() for m in os.getenv("OPENAI_FALLBACK_MODELS", "gpt-4.1,gpt-4o").split(",") if m.strip()]
OPENAI_API_STYLE = os.getenv("OPENAI_API_STYLE", "responses").strip().lower()
OPENAI_URL = os.getenv("OPENAI_URL", "https://api.openai.com/v1/responses")

LLM_JUDGE_PROMPT = """Eres auditor de entity resolution de empresas en Ecuador.
Tu tarea es SOLO decidir si el RUC candidato provisto corresponde plausiblemente al nombre fuente.
No inventes RUC, no cambies RUC, no completes datos faltantes.
Reglas:
- Si el RUC no tiene exactamente 13 dígitos, verdict=incorrect.
- Si candidate_norm es genérico o demasiado parcial frente a source_norm, verdict=incorrect o uncertain.
- Si source_norm y candidate_norm representan claramente la misma razón social o nombre comercial local, verdict=correct.
- Si hay duda razonable, verdict=uncertain.
Devuelve JSON estricto con la forma: {"results":[{"id":"...","verdict":"correct|incorrect|uncertain","confidence":0.0,"reason":"frase corta"}]}.
"""

LLM_RESPONSE_SCHEMA = {
    "type": "object",
    "additionalProperties": False,
    "properties": {
        "results": {
            "type": "array",
            "items": {
                "type": "object",
                "additionalProperties": False,
                "properties": {
                    "id": {"type": "string"},
                    "verdict": {"type": "string", "enum": ["correct", "incorrect", "uncertain"]},
                    "confidence": {"type": "number"},
                    "reason": {"type": "string"},
                },
                "required": ["id", "verdict", "confidence", "reason"],
            },
        }
    },
    "required": ["results"],
}


def _extract_json(text: str) -> str:
    s = str(text or "").strip()
    if s.startswith("```"):
        s = s.split("\n", 1)[1].rsplit("```", 1)[0].strip()
    i, j = s.find("{"), s.rfind("}")
    return s[i:j+1] if i != -1 and j != -1 else s


def _response_output_text(obj: dict) -> str:
    if obj.get("output_text"):
        return str(obj["output_text"])
    chunks = []
    for item in obj.get("output", []) or []:
        for content in item.get("content", []) or []:
            if isinstance(content, dict) and "text" in content:
                chunks.append(str(content["text"]))
    return "\n".join(chunks)


def _post_json(url: str, payload: dict) -> dict:
    data = json.dumps(payload).encode("utf-8")
    req = urllib.request.Request(
        url,
        data=data,
        headers={"Content-Type": "application/json", "Authorization": f"Bearer {OPENAI_API_KEY}"},
        method="POST",
    )
    with urllib.request.urlopen(req, timeout=300) as resp:
        raw = resp.read().decode("utf-8")
    return json.loads(raw)


def _call_llm_responses(chunk: list, model: str) -> list:
    payload = {
        "model": model,
        "input": [
            {"role": "system", "content": LLM_JUDGE_PROMPT},
            {"role": "user", "content": json.dumps({"cases": chunk}, ensure_ascii=False)},
        ],
        "max_output_tokens": int(os.getenv("OPENAI_MAX_TOKENS", "2000")),
        "text": {
            "format": {
                "type": "json_schema",
                "name": "ruc_audit_results",
                "schema": LLM_RESPONSE_SCHEMA,
                "strict": True,
            }
        },
    }
    obj = _post_json(OPENAI_URL, payload)
    content = _extract_json(_response_output_text(obj))
    return json.loads(content).get("results", [])


def _call_llm_chat(chunk: list, model: str) -> list:
    chat_url = os.getenv("OPENAI_CHAT_URL", "https://api.openai.com/v1/chat/completions")
    payload = {
        "model": model,
        "temperature": 0,
        "max_tokens": int(os.getenv("OPENAI_MAX_TOKENS", "2000")),
        "response_format": {"type": "json_object"},
        "messages": [
            {"role": "system", "content": LLM_JUDGE_PROMPT},
            {"role": "user", "content": json.dumps({"cases": chunk}, ensure_ascii=False)},
        ],
    }
    obj = _post_json(chat_url, payload)
    content = _extract_json(obj["choices"][0]["message"]["content"])
    return json.loads(content).get("results", [])


def _call_llm(chunk: list, model: str) -> list:
    if OPENAI_API_STYLE == "chat":
        return _call_llm_chat(chunk, model)
    return _call_llm_responses(chunk, model)


def _judge_all(cases: list) -> pd.DataFrame:
    if not cases:
        return pd.DataFrame(columns=["id", "verdict", "confidence", "reason", "llm_model"])
    models = [OPENAI_MODEL] + [m for m in OPENAI_FALLBACK_MODELS if m != OPENAI_MODEL]
    chunk_size = int(os.getenv("AUDIT_CHUNK_SIZE", "10"))
    results = []
    for start in range(0, len(cases), chunk_size):
        chunk = cases[start:start + chunk_size]
        last_error = None
        for model in models:
            try:
                chunk_results = _call_llm(chunk, model)
                for r in chunk_results:
                    r["llm_model"] = model
                results.extend(chunk_results)
                last_error = None
                break
            except Exception as e:
                last_error = e
        if last_error is not None:
            print(f"[WARN] Falló auditoría LLM para chunk {start}: {last_error}")
            for case in chunk:
                results.append({
                    "id": case["id"],
                    "verdict": "uncertain",
                    "confidence": 0.0,
                    "reason": "fallo llamada llm",
                    "llm_model": "",
                })
    return pd.DataFrame(results)


if audit_cases and OPENAI_API_KEY:
    print(f"Auditando {len(audit_cases)} casos con {OPENAI_MODEL} vía {OPENAI_API_STYLE}...")
    judge_df = _judge_all(audit_cases)
else:
    print("[INFO] Sin API key o sin casos: no se llama al LLM. Fuzzy no exacto queda para revisión.")
    judge_df = pd.DataFrame(columns=["id", "verdict", "confidence", "reason", "llm_model"])

AUDIT_OUT_COLS = ["id", "source_label", "source_raw", "source_norm", "candidate_norm", "source_winner", "score", "ruc", "razon_social_ecuador_raw", "usa_razon_social_validada", "verdict", "confidence", "reason", "llm_model"]
cases_df = pd.DataFrame(audit_cases)
out_df = cases_df.merge(judge_df, on="id", how="left") if not cases_df.empty else pd.DataFrame(columns=AUDIT_OUT_COLS)
for c in AUDIT_OUT_COLS:
    if c not in out_df.columns:
        out_df[c] = pd.NA
out_df = out_df[AUDIT_OUT_COLS]
if not out_df.empty:
    out_df["verdict"] = out_df["verdict"].fillna("not_audited")
    out_df["confidence"] = pd.to_numeric(out_df["confidence"], errors="coerce")
    print(out_df["verdict"].value_counts(dropna=False).to_string())
_ = save_df_csv(out_df, CLEAN_OUT / "audit_df.csv")


[INFO] Sin API key o sin casos: no se llama al LLM. Fuzzy no exacto queda para revisión.
verdict
not_audited    37
[OK] Guardado: E:\TESIS MAESTRIA\Desarrollo_clustering_maestria\02_data_cleaning\outputs\audit_df.csv  shape: (37, 14)


In [5]:
# ── Match final conservador ─────────────────────────────────────────────────
def final_from_exact(exact_df: pd.DataFrame, *, label: str, raw_col: str, norm_col: str) -> pd.DataFrame:
    d = exact_df.copy()
    d["RUC"] = d["RUC"].map(normalize_ruc)
    d = d[d["RUC"].map(is_valid_ruc)].copy()
    if d.empty:
        return pd.DataFrame(columns=COLS_FINAL_BASE)
    d["source_label"] = label
    d["name_raw"] = d[raw_col]
    d["name_norm"] = d[norm_col]
    d["source_winner"] = "SCVS_EXACT"
    d["score"] = 100.0
    d["verdict"] = "correct"
    d["evidence_type"] = "exact_scvs"
    if f"{norm_col}_original" in d.columns:
        d["name_norm_original"] = d[f"{norm_col}_original"]
    else:
        d["name_norm_original"] = d["name_norm"]
    match_col = "Company_match_raw" if label == "LEADS" else "EMPRESA_match_raw"
    d["name_match_raw"] = d[match_col] if match_col in d.columns else d["name_raw"]
    for c in ["razon_social_ecuador_raw"]:
        if c not in d.columns:
            d[c] = ""
    for c in ["usa_razon_social_validada", "sede_ecuador", "validacion_manual"]:
        if c not in d.columns:
            d[c] = False
        d[c] = d[c].map(to_bool)
    return d[COLS_FINAL_BASE]


def final_from_manual_razon_exact(manual_df: pd.DataFrame, *, label: str, raw_col: str, norm_col: str) -> pd.DataFrame:
    d = manual_df.copy()
    if d.empty or "RUC" not in d.columns:
        return pd.DataFrame(columns=COLS_FINAL_BASE)
    d["RUC"] = d["RUC"].map(normalize_ruc)
    if "manual_match_status" in d.columns:
        d = d[d["manual_match_status"].eq("MATCHED")].copy()
    d = d[d["RUC"].map(is_valid_ruc)].copy()
    if d.empty:
        return pd.DataFrame(columns=COLS_FINAL_BASE)
    d["source_label"] = label
    d["name_raw"] = d[raw_col]
    d["name_norm"] = d[norm_col]
    d["source_winner"] = d["source_winner"] if "source_winner" in d.columns else d.get("manual_match_method", "MANUAL_RAZON")
    d["score"] = pd.to_numeric(d["score"], errors="coerce").fillna(100.0) if "score" in d.columns else 100.0
    d["verdict"] = "correct"
    d["evidence_type"] = "manual_razon_social_exact"
    if f"{norm_col}_original" in d.columns:
        d["name_norm_original"] = d[f"{norm_col}_original"]
    else:
        d["name_norm_original"] = d["name_norm"]
    match_col = "Company_match_raw" if label == "LEADS" else "EMPRESA_match_raw"
    d["name_match_raw"] = d[match_col] if match_col in d.columns else d.get("razon_social_ecuador_raw", d["name_raw"])
    for c in ["razon_social_ecuador_raw"]:
        if c not in d.columns:
            d[c] = ""
    for c in ["usa_razon_social_validada", "sede_ecuador", "validacion_manual"]:
        if c not in d.columns:
            d[c] = False
        d[c] = d[c].map(to_bool)
    return d[COLS_FINAL_BASE]


def accepted_from_candidates(candidates: pd.DataFrame, audit_df_: pd.DataFrame) -> pd.DataFrame:
    if candidates.empty:
        return pd.DataFrame(columns=COLS_FINAL_BASE)
    d = candidates.copy()
    d["RUC"] = d["RUC"].map(normalize_ruc)
    if not audit_df_.empty and "id" in audit_df_.columns:
        verdicts = audit_df_[["id", "verdict", "confidence", "reason"]].rename(columns={"id": "audit_id", "verdict": "llm_verdict", "confidence": "llm_confidence", "reason": "llm_reason"})
        d = d.merge(verdicts, on="audit_id", how="left")
    else:
        d["llm_verdict"] = "not_audited"
        d["llm_confidence"] = pd.NA
        d["llm_reason"] = "no api key"

    d["llm_confidence"] = pd.to_numeric(d["llm_confidence"], errors="coerce")
    exact_name_ok = d["deterministic_decision"].eq("accept_exact_name")
    llm_ok = (
        d["deterministic_decision"].eq("needs_llm")
        & d["llm_verdict"].eq("correct")
        & (d["llm_confidence"].fillna(0) >= AUDIT_MIN_CONFIDENCE)
    )
    no_llm_override = ALLOW_FUZZY_WITHOUT_LLM & d["deterministic_decision"].eq("needs_llm")
    accepted_mask = d["RUC"].map(is_valid_ruc) & (exact_name_ok | llm_ok | no_llm_override)
    accepted = d[accepted_mask].copy()
    if accepted.empty:
        return pd.DataFrame(columns=COLS_FINAL_BASE)

    accepted["source_winner"] = accepted["source_winner"].astype(str) + "_TORNEO"
    accepted["verdict"] = "correct"
    accepted.loc[llm_ok.loc[accepted.index], "verdict"] = "llm_correct"
    accepted.loc[no_llm_override.loc[accepted.index], "verdict"] = "accepted_without_llm"
    accepted["evidence_type"] = "torneo_llm_correct"
    accepted.loc[exact_name_ok.loc[accepted.index], "evidence_type"] = "torneo_exact_name"
    accepted.loc[no_llm_override.loc[accepted.index], "evidence_type"] = "torneo_fuzzy_without_llm"
    return accepted[COLS_FINAL_BASE]

exact_final = pd.concat([
    final_from_exact(leads_exact, label="LEADS", raw_col="Company_raw", norm_col="Company_norm"),
    final_from_exact(horas_exact, label="HORAS", raw_col="EMPRESA_raw", norm_col="EMPRESA_norm"),
], ignore_index=True)

manual_final = pd.concat([
    final_from_manual_razon_exact(leads_manual_exact, label="LEADS", raw_col="Company_raw", norm_col="Company_norm"),
    final_from_manual_razon_exact(horas_manual_exact, label="HORAS", raw_col="EMPRESA_raw", norm_col="EMPRESA_norm"),
], ignore_index=True)

fuzzy_final = accepted_from_candidates(candidates_df, out_df)
accepted_all = pd.concat([exact_final, manual_final, fuzzy_final], ignore_index=True)
match_final = (
    accepted_all
    .drop_duplicates(subset=["name_norm", "RUC"], keep="first")
    .sort_values(["name_norm", "source_label"])
    .reset_index(drop=True)
)

if not universe.empty and "name_norm" in universe.columns:
    extra_cols = [c for c in universe.columns if c != "name_norm"]
    match_final = match_final.merge(universe[["name_norm", *extra_cols]], on="name_norm", how="left", suffixes=("", "_universe"))

accepted_keys = set(zip(accepted_all["source_label"], accepted_all["name_raw"], accepted_all["name_norm"])) if not accepted_all.empty else set()
all_sources = []
for exact_df, label, raw_col, norm_col in [
    (leads_exact, "LEADS", "Company_raw", "Company_norm"),
    (horas_exact, "HORAS", "EMPRESA_raw", "EMPRESA_norm"),
]:
    d = exact_df.copy()
    d["source_label"] = label
    d["name_raw"] = d[raw_col]
    d["name_norm"] = d[norm_col]
    d["RUC"] = d["RUC"].map(normalize_ruc)
    d["ruc_valido"] = d["RUC"].map(is_valid_ruc)
    for c in ["usa_razon_social_validada", "sede_ecuador", "validacion_manual"]:
        if c not in d.columns:
            d[c] = False
        d[c] = d[c].map(to_bool)
    all_sources.append(d)
all_sources_df = pd.concat(all_sources, ignore_index=True)
not_accepted = all_sources_df[~all_sources_df.apply(lambda r: (r["source_label"], r["name_raw"], r["name_norm"]) in accepted_keys, axis=1)].copy()
not_accepted["estado_revision"] = "SIN_RUC_NO_VALIDADO"
not_accepted.loc[not_accepted["sede_ecuador"].eq(False), "estado_revision"] = "DESCARTAR_SIN_SEDE_ECUADOR_MANUAL"
not_accepted.loc[not_accepted["usa_razon_social_validada"].eq(True) & not_accepted["sede_ecuador"].eq(True), "estado_revision"] = "REVISION_RUC_RAZON_SOCIAL_MANUAL"

review_cols = [
    "audit_id", "source_label", "name_raw", "name_norm", "name_norm_torneo_original", "winner_name_norm", "RUC", "source_winner", "score",
    "deterministic_decision", "deterministic_reason", "llm_verdict", "llm_confidence", "llm_reason",
    "razon_social_ecuador_raw", "usa_razon_social_validada", "sede_ecuador", "validacion_manual",
]
review_df = candidates_df.copy()
if not out_df.empty:
    verdicts = out_df[["id", "verdict", "confidence", "reason"]].rename(columns={"id": "audit_id", "verdict": "llm_verdict", "confidence": "llm_confidence", "reason": "llm_reason"})
    review_df = review_df.merge(verdicts, on="audit_id", how="left")
for c in review_cols:
    if c not in review_df.columns:
        review_df[c] = pd.NA
review_df = review_df[review_cols]

print(f"[MATCH FINAL] Total aceptados: {len(match_final)}")
print(match_final["evidence_type"].value_counts(dropna=False).to_string() if not match_final.empty else "")
print(f"[REVISION] Candidatos torneo: {len(review_df)} | No aceptados base: {len(not_accepted)}")

_ = save_df_csv(review_df, CLEAN_OUT / "candidatos_revision_empresas.csv")
_ = save_df_csv(not_accepted, CLEAN_OUT / "empresas_sin_match_final.csv")
_ = save_df_csv(match_final, CLEAN_OUT / "match_final_empresas.csv")

print("\n✓ Notebook 04_auditoria_llm_y_match_final completado correctamente.")
display(match_final.head(10))


[MATCH FINAL] Total aceptados: 153
evidence_type
exact_scvs                   128
manual_razon_social_exact     25
[REVISION] Candidatos torneo: 177 | No aceptados base: 267
[OK] Guardado: E:\TESIS MAESTRIA\Desarrollo_clustering_maestria\02_data_cleaning\outputs\candidatos_revision_empresas.csv  shape: (177, 18)
[OK] Guardado: E:\TESIS MAESTRIA\Desarrollo_clustering_maestria\02_data_cleaning\outputs\empresas_sin_match_final.csv  shape: (267, 21)
[OK] Guardado: E:\TESIS MAESTRIA\Desarrollo_clustering_maestria\02_data_cleaning\outputs\match_final_empresas.csv  shape: (153, 24)

✓ Notebook 04_auditoria_llm_y_match_final completado correctamente.


C:\Users\asus\AppData\Local\Temp\ipykernel_17824\3938078376.py:110: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  accepted_all = pd.concat([exact_final, manual_final, fuzzy_final], ignore_index=True)


,source_label,name_raw,name_norm,RUC,source_winner,score,verdict,evidence_type,razon_social_ecuador_raw,usa_razon_social_validada,...,name_raw_universe,sources,n_rows,name_norm_antes_limpieza,name_raw_antes_limpieza,sources_antes_limpieza,n_rows_antes_limpieza,name_match_raw_universe,razon_social_ecuador_raw_universe,usa_razon_social_validada_universe
0,LEADS,ABBOTT,ABBOTT LABORATORIOS ECUADOR,0990000670001,SCVS_EXACT,100.0,correct,exact_scvs,ABBOTT LABORATORIOS DEL ECUADOR CIA. LTDA.,True,...,ABBOTT,LEADS,1,ABBOTT,ABBOTT,LEADS,1,ABBOTT LABORATORIOS DEL ECUADOR CIA. LTDA.,ABBOTT LABORATORIOS DEL ECUADOR CIA. LTDA.,True
1,LEADS,Abbvie,ABBVIE,1792489156001,MANUAL_RAZON_SRI_LEGAL,100.0,correct,manual_razon_social_exact,ABBVIE S.A.S.,True,...,Abbvie,LEADS,1,ABBVIE,Abbvie,LEADS,1,ABBVIE S.A.S.,ABBVIE S.A.S.,True
2,LEADS,AIG,AIG METROPOLITANA SEGUROS REASEGUROS,1790475247001,SCVS_EXACT,100.0,correct,exact_scvs,AIG METROPOLITANA CIA. DE SEGUROS Y REASEGUROS...,True,...,AIG,LEADS,1,AIG,AIG,LEADS,1,AIG METROPOLITANA CIA. DE SEGUROS Y REASEGUROS...,AIG METROPOLITANA CIA. DE SEGUROS Y REASEGUROS...,True
3,LEADS,Akros,AKROS,1791148800001,SCVS_EXACT,100.0,correct,exact_scvs,AKROS CIA. LTDA.,True,...,Akros,LEADS,1,AKROS,Akros,LEADS,1,AKROS CIA. LTDA.,AKROS CIA. LTDA.,True
4,LEADS,ALCIONE MX,ALCIONEO,0993373523001,SCVS_EXACT,100.0,correct,exact_scvs,Alcioneo S.A.S.,True,...,ALCIONE MX,LEADS,1,ALCIONE MX,ALCIONE MX,LEADS,1,Alcioneo S.A.S.,Alcioneo S.A.S.,True
5,HORAS,El Sabor,ALIMENTOS SABOR ALIMENSABOR,0990294690001,SCVS_EXACT,100.0,correct,exact_scvs,ALIMENTOS EL SABOR ALIMENSABOR C.LTDA..,True,...,El Sabor,HORAS,1,SABOR,El Sabor,HORAS,1,ALIMENTOS EL SABOR ALIMENSABOR C.LTDA..,ALIMENTOS EL SABOR ALIMENSABOR C.LTDA..,True
6,HORAS,DePrati,ALMACENES PRATI,0990011214001,SCVS_EXACT,100.0,correct,exact_scvs,ALMACENES DE PRATI S.A..,True,...,DePrati,HORAS,1,DEPRATI,DePrati,HORAS,1,ALMACENES DE PRATI S.A..,ALMACENES DE PRATI S.A..,True
7,LEADS,Alpina,ALPINA PRODUCTOS ALIMENTICIOS ALPIECUADOR,1791302400001,SCVS_EXACT,100.0,correct,exact_scvs,ALPINA PRODUCTOS ALIMENTICIOS ALPIECUADOR S.A..,True,...,Alpina,LEADS,1,ALPINA,Alpina,LEADS,1,ALPINA PRODUCTOS ALIMENTICIOS ALPIECUADOR S.A..,ALPINA PRODUCTOS ALIMENTICIOS ALPIECUADOR S.A..,True
8,LEADS,Alvarez Barba S.A,ALVAREZ BARBA,1790360741001,SCVS_EXACT,100.0,correct,exact_scvs,Alvarez Barba S.A,True,...,Alvarez Barba S.A,LEADS,1,ALVAREZ BARBA,Alvarez Barba S.A,LEADS,1,Alvarez Barba S.A,Alvarez Barba S.A,True
9,LEADS,AMBROSIA,AMBROSIA WINESHOP,1793141250001,SCVS_EXACT,100.0,correct,exact_scvs,Ambrosia Wineshop S.A.S..,True,...,AMBROSIA,LEADS,1,AMBROSIA,AMBROSIA,LEADS,1,Ambrosia Wineshop S.A.S..,Ambrosia Wineshop S.A.S..,True
